# 面试问题：PagedAttention 的 KV Block Manager 怎样降低预留浪费，并安全处理增长和取消？

        ## 可直接复述的回答主线

        1. 传统连续 KV 预留按最大输出长度占位，短请求会长期占着尚未使用的显存。
2. PagedAttention 把逻辑 Token 位置映射到固定大小物理块，请求只在跨块边界时增量申请。
3. Block Manager 必须维护 free list、每请求 block table、Token 计数和唯一所有权。
4. 比较方案时要展示准入数、实际块数、预留浪费和事件级占用，而不能只验证最终 free 数。
5. 取消请求必须幂等释放其全部块，否则长时间 Serving 会出现隐蔽 KV 泄漏。
6. 生产实现还需处理并发、前缀共享、Copy-on-Write、设备同步和调度公平性。

        后续实验会用同一批输入依次验证朴素方案、核心机制、失败边界和修正效果。

## 1. 真实案例与输入预览

案例是六条中文客服生成请求，每条包含 prompt Token、最大生成 Token、deadline 和取消语义。物理池只有 18 个块、每块 4 Token，故意让连续最大长度预留发生拒绝，而按需分页可以全部准入。事件为离线确定性样本，不代表真实吞吐。

In [1]:
import copy  # 复制块管理器以隔离泄漏失败实验。
import math  # 计算 Token 数对应的物理块数量。
requests = [{"id": "req-01", "queue": "退款", "prompt_tokens": 7, "max_new_tokens": 6, "deadline_ms": 900}, {"id": "req-02", "queue": "物流", "prompt_tokens": 5, "max_new_tokens": 8, "deadline_ms": 1100}, {"id": "req-03", "queue": "售后", "prompt_tokens": 11, "max_new_tokens": 3, "deadline_ms": 850}, {"id": "req-04", "queue": "会员", "prompt_tokens": 3, "max_new_tokens": 9, "deadline_ms": 1400}, {"id": "req-05", "queue": "发票", "prompt_tokens": 8, "max_new_tokens": 4, "deadline_ms": 1000}, {"id": "req-06", "queue": "投诉", "prompt_tokens": 4, "max_new_tokens": 5, "deadline_ms": 700}]  # 定义六条具有 Serving 字段的脱敏请求。
block_size = 4  # 设定每个 KV 物理块能够容纳四个 Token。
block_capacity = 18  # 设定教学池总共只有十八个物理块。
print("教学实验输入：KV 块池容量=18，块大小=4 Token")  # 输出资源上限供后续准入对照。
print("请求      队列  prompt  max_new  deadline")  # 输出请求预览表头。
for request in requests:  # 逐条展示六条请求的真实调度字段。
    print(f"{request['id']:<9} {request['queue']:<2} {request['prompt_tokens']:>7} {request['max_new_tokens']:>8} {request['deadline_ms']:>8}ms")  # 输出当前请求字段。

教学实验输入：KV 块池容量=18，块大小=4 Token
请求      队列  prompt  max_new  deadline
req-01    退款       7        6      900ms
req-02    物流       5        8     1100ms
req-03    售后      11        3      850ms
req-04    会员       3        9     1400ms
req-05    发票       8        4     1000ms
req-06    投诉       4        5      700ms


## 2. Baseline / 基线：按 prompt 加最大输出连续预留

基线在准入时一次保留最坏长度，直到请求结束才归还。它实现简单，却把尚未生成的 Token 也计入占用，导致后到请求被拒绝。

In [2]:
reserved_total = 0  # 记录连续预留策略已经承诺的块数。
baseline_rows = []  # 保存每条请求的预留和准入结果。
for request in requests:  # 按到达顺序执行朴素准入。
    reserved = math.ceil((request["prompt_tokens"] + request["max_new_tokens"]) / block_size)  # 按最坏总长度计算连续预留块。
    admitted = reserved_total + reserved <= block_capacity  # 检查加入当前请求后是否超过池容量。
    reserved_total += reserved if admitted else 0  # 只为成功准入请求提交预留。
    baseline_rows.append({"id": request["id"], "reserved": reserved, "admitted": admitted, "pool_used": reserved_total})  # 保存当前决策和池占用。
print("Baseline 连续预留账本")  # 标记当前输出属于最大长度预留。
print("请求      预留块  准入  累计占用")  # 输出准入账本表头。
for row in baseline_rows:  # 逐请求展示预留策略的决策。
    print(f"{row['id']:<9} {row['reserved']:>6} {str(row['admitted']):>6} {row['pool_used']:>9}")  # 输出当前请求的预留和累计占用。
print("Baseline 准入数=", sum(row["admitted"] for row in baseline_rows))  # 展示连续预留造成的容量损失。

Baseline 连续预留账本
请求      预留块  准入  累计占用
req-01         4   True         4
req-02         4   True         8
req-03         4   True        12
req-04         3   True        15
req-05         3   True        18
req-06         3  False        18
Baseline 准入数= 5


## 3. 底层实现：free list、block table 与增量跨块申请

下面用标准库实现最小 Block Manager。每个物理块只有一个 owner，请求在 prompt 准入和 Token 跨边界时申请，结束或取消时统一 release。

In [3]:
class BlockManager:  # 实现具有唯一所有权的最小 KV 物理块管理器。
    def __init__(self, capacity, tokens_per_block):  # 初始化物理池和每请求元数据。
        self.capacity = capacity  # 保存总物理块数量供容量审计。
        self.tokens_per_block = tokens_per_block  # 保存每块 Token 容量供增长判断。
        self.free = list(range(capacity))  # 用有序 free list 保证教学输出确定。
        self.tables = {}  # 保存 request_id 到物理块序列的映射。
        self.tokens = {}  # 保存每个活动请求当前 Token 数。
    def admit(self, request):  # 为新请求按 prompt 实际长度分配初始块。
        needed = math.ceil(request["prompt_tokens"] / self.tokens_per_block)  # 计算 prompt 当前真正需要的块数。
        if needed > len(self.free):  # 在容量不足时拒绝部分分配。
            return False  # 返回明确准入失败而不污染块表。
        blocks = [self.free.pop(0) for _ in range(needed)]  # 从 free list 取得确定性物理块。
        self.tables[request["id"]] = blocks  # 提交该请求的逻辑到物理映射。
        self.tokens[request["id"]] = request["prompt_tokens"]  # 记录 prompt 已占用的 Token 数。
        return True  # 告知调度器准入成功。
    def append_token(self, request_id):  # 为运行请求追加一个生成 Token。
        if self.tokens[request_id] % self.tokens_per_block == 0:  # 仅在当前末块已经填满时申请新块。
            if not self.free:  # 检查物理池是否仍有空闲块。
                return False  # 容量不足时不提交 Token。
            self.tables[request_id].append(self.free.pop(0))  # 把新物理块追加到请求 block table。
        self.tokens[request_id] += 1  # 提交新增 Token 的逻辑长度。
        return True  # 告知解码器本次增长成功。
    def release(self, request_id):  # 幂等释放请求拥有的全部物理块。
        blocks = self.tables.pop(request_id, [])  # 原子取走 block table，重复调用得到空列表。
        self.tokens.pop(request_id, None)  # 删除请求 Token 计数且允许重复释放。
        self.free.extend(blocks)  # 把请求全部物理块归还 free list。
        self.free.sort()  # 恢复确定顺序便于复现和审计。
        return blocks  # 返回实际释放块用于事件账本。
    def owner_map(self):  # 生成物理块到唯一请求的反向索引。
        return {block: request_id for request_id, blocks in self.tables.items() for block in blocks}  # 展开所有活动 block table。
manager = BlockManager(block_capacity, block_size)  # 创建按需分页 KV 管理器。
paged_admissions = [manager.admit(request) for request in requests]  # 仅按六条 prompt 的实际长度执行准入。
print("PagedAttention 初始 block table")  # 标记下表是按需物理映射。
for request in requests:  # 逐请求展示逻辑长度和物理块序列。
    print(f"{request['id']} tokens={manager.tokens.get(request['id'])} blocks={manager.tables.get(request['id'])}")  # 输出当前请求 block table。
print(f"全部准入={all(paged_admissions)}，实际占用={block_capacity - len(manager.free)}块，剩余={len(manager.free)}块")  # 展示按需分页的初始容量结果。

PagedAttention 初始 block table
req-01 tokens=7 blocks=[0, 1]
req-02 tokens=5 blocks=[2, 3]
req-03 tokens=11 blocks=[4, 5, 6]
req-04 tokens=3 blocks=[7]
req-05 tokens=8 blocks=[8, 9]
req-06 tokens=4 blocks=[10]
全部准入=True，实际占用=11块，剩余=7块


## 4. 结果表与结果解读

执行一轮 Decode 后，只有跨越块边界的请求才增加物理块。随后取消 req-02 并展示释放事件，形成可读的调度与资源账本。

In [4]:
event_ledger = []  # 收集每个请求追加 Token 后的资源变化。
for request in requests:  # 为六个活动请求各执行一次确定性 Decode。
    before_blocks = len(manager.tables[request["id"]])  # 记录追加前的物理块数量。
    appended = manager.append_token(request["id"])  # 尝试为当前请求提交一个 Token。
    after_blocks = len(manager.tables[request["id"]])  # 记录追加后的物理块数量。
    event_ledger.append({"id": request["id"], "appended": appended, "new_block": after_blocks > before_blocks, "blocks": after_blocks, "free": len(manager.free)})  # 保存跨块和剩余容量。
released_req02 = manager.release("req-02")  # 模拟客户端取消并幂等释放其 KV。
print("一轮 Decode 资源事件账本")  # 标记下表展示动态增长而非静态预留。
print("请求      追加成功  新块  当前块数  剩余块")  # 输出事件账本表头。
for event in event_ledger:  # 逐条展示 Token 增长与块分配。
    print(f"{event['id']:<9} {str(event['appended']):>8} {str(event['new_block']):>5} {event['blocks']:>9} {event['free']:>7}")  # 输出当前请求资源变化。
print(f"取消 req-02 释放块={released_req02}，取消后剩余={len(manager.free)}块")  # 展示取消带来的资源回收。
print(f"解读：连续预留只准入 {sum(row['admitted'] for row in baseline_rows)} 条，分页初始准入 {sum(paged_admissions)} 条，且只为真实增长付费。")  # 给出同池容量下的直接结论。

一轮 Decode 资源事件账本
请求      追加成功  新块  当前块数  剩余块
req-01        True False         2       7
req-02        True False         2       7
req-03        True False         3       7
req-04        True False         1       7
req-05        True  True         3       6
req-06        True  True         2       5
取消 req-02 释放块=[2, 3]，取消后剩余=7块
解读：连续预留只准入 5 条，分页初始准入 6 条，且只为真实增长付费。


## 5. 失败案例与修正

常见失败是取消时只删除请求状态，没有归还 block table。这里在副本中制造 req-05 泄漏，再调用统一 release 修正，并观察 free list 的变化。

In [5]:
leak_manager = copy.deepcopy(manager)  # 复制当前管理器以隔离故意制造的泄漏。
free_before_leak = len(leak_manager.free)  # 记录错误取消前的空闲块数。
leak_manager.tokens.pop("req-05", None)  # 模拟错误路径只删除逻辑请求状态。
leaked_blocks = list(leak_manager.tables["req-05"])  # 找出仍被 block table 占有的物理块。
free_after_wrong_cancel = len(leak_manager.free)  # 记录错误取消后没有变化的空闲块数。
recovered_blocks = leak_manager.release("req-05")  # 通过统一幂等释放路径修正泄漏。
print(f"错误行为：取消 req-05 后 free={free_after_wrong_cancel}，仍泄漏块={leaked_blocks}")  # 展示幽灵所有权和未恢复容量。
print(f"修正行为：release 归还={recovered_blocks}，free 从 {free_before_leak} 增至 {len(leak_manager.free)}")  # 展示修正后的容量恢复。

错误行为：取消 req-05 后 free=7，仍泄漏块=[8, 9, 11]
修正行为：release 归还=[8, 9, 11]，free 从 7 增至 10


## 6. 生产边界

教学管理器没有并发锁、GPU stream 安全点、前缀共享引用计数、Copy-on-Write、跨 worker 迁移和优先级抢占。生产系统必须把调度状态与设备完成事件绑定，避免仍在 kernel 使用时提前释放。

In [6]:
production_metrics = {"active_requests": len(manager.tables), "owned_blocks": len(manager.owner_map()), "free_blocks": len(manager.free), "capacity": manager.capacity}  # 汇总生产监控最小低基数指标。
print("资源审计快照：", production_metrics)  # 输出可用于发现泄漏和容量漂移的监控字段。

资源审计快照： {'active_requests': 5, 'owned_blocks': 11, 'free_blocks': 7, 'capacity': 18}


## 7. 最小回归测试

只验证样本规模、分页准入、唯一所有权、幂等释放和泄漏修正。

In [7]:
assert len(requests) >= 5  # 保证案例包含至少五条有业务字段的请求。
assert all(paged_admissions)  # 保证按 prompt 实际长度时六条请求都能准入。
assert len(manager.owner_map()) == len(set(manager.owner_map()))  # 保证每个物理块只有一个活动 owner。
assert manager.release("req-02") == []  # 保证重复取消不会二次归还同一物理块。
assert recovered_blocks == leaked_blocks  # 保证修正路径找回故意泄漏的全部块。